# **团队成员：**
# | 姓名 | 学号 |
# | 郑泽凯 | 2025416010129 |
# | 邱婕芹 | 2025416010119 |
# | 何想 | 2025416010106 |

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# %% [markdown]
# # 🏪 Store Sales - Time Series Forecasting (Final High-Score Version)
# %%
import numpy as np
import pandas as pd
import lightgbm as lgb
import os
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_log_error
import warnings
warnings.filterwarnings('ignore')

# ---- 1. 动态寻路 ----
DATA_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train.csv' in files:
        DATA_DIR = Path(root)
        break
if DATA_DIR is None:
    raise FileNotFoundError("❌ 未能在 /kaggle/input 下找到数据文件。")

train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['date'])
test = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=['date'])
stores = pd.read_csv(DATA_DIR / 'stores.csv')
oil = pd.read_csv(DATA_DIR / 'oil.csv', parse_dates=['date'])
holidays = pd.read_csv(DATA_DIR / 'holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv(DATA_DIR / 'transactions.csv', parse_dates=['date'])

# ---- 2. 脏数据截断 & 圣诞节特殊闭店处理 ----
train = train[train['date'] >= '2015-01-01'].reset_index(drop=True)
train.loc[(train['date'].dt.month == 12) & (train['date'].dt.day == 25), 'sales'] = 0.0

# ---- 3. 黄金协变量前置完整处理 ----
oil = oil.rename(columns={'dcoilwtico': 'oil_price'})
oil['oil_price'] = oil['oil_price'].interpolate(method='linear').bfill().ffill()

# 复活：油价多阶变动特征
oil['oil_change_1d'] = oil['oil_price'].pct_change(1)
oil['oil_change_3d'] = oil['oil_price'].pct_change(3)
oil['oil_change_7d'] = oil['oil_price'].pct_change(7)
oil['oil_ma_7'] = oil['oil_price'].rolling(7, min_periods=1).mean()
oil['oil_ma_28'] = oil['oil_price'].rolling(28, min_periods=1).mean()
oil['oil_diff_ma7'] = oil['oil_price'] - oil['oil_ma_7']

# 交易量安全右移16天
shifted_tx = transactions.copy()
shifted_tx['date'] = shifted_tx['date'] + pd.Timedelta(days=16)
shifted_tx = shifted_tx.rename(columns={'transactions': 'store_transactions'})

# 完整复活：多级节假日精细特征
holidays['is_transferred'] = (holidays['transferred'] == True).astype(int)
nat_hol = holidays[(holidays['locale'] == 'National') & (holidays['transferred'] == False)].drop_duplicates('date')
nat_hol = nat_hol.rename(columns={'type': 'holiday_type_nat'})
nat_hol['is_national_holiday'] = 1

reg_hol = holidays[(holidays['locale'] == 'Regional') & (holidays['transferred'] == False)].drop_duplicates(['date', 'locale_name'])
reg_hol = reg_hol.rename(columns={'type': 'holiday_type_reg', 'locale_name': 'state'})
reg_hol['is_regional_holiday'] = 1

loc_hol = holidays[(holidays['locale'] == 'Local') & (holidays['transferred'] == False)].drop_duplicates(['date', 'locale_name'])
loc_hol = loc_hol.rename(columns={'type': 'holiday_type_loc', 'locale_name': 'city'})
loc_hol['is_local_holiday'] = 1

# 发薪日连续特征流
all_dates = pd.date_range(train['date'].min(), test['date'].max())
paydays = all_dates[(all_dates.day == 15) | (all_dates.is_month_end)]
oil_only_dates = pd.DataFrame({'date': all_dates})
oil_only_dates['days_since_payday'] = oil_only_dates['date'].apply(
    lambda x: min([(x - p).days for p in paydays if (x - p).days >= 0], default=0)
)
oil_only_dates['is_payday'] = ((oil_only_dates['date'].dt.day == 15) | oil_only_dates['date'].dt.is_month_end).astype(int)

earthquake_date = pd.Timestamp('2016-04-16')

# ---- 4. 统一特征工程管道（完美重塑你们的特征库） ----
def build_all_features(df):
    df = df.copy()
    df = df.merge(stores, on='store_nbr', how='left')
    df = df.merge(oil, on='date', how='left')
    df = df.merge(shifted_tx, on=['date', 'store_nbr'], how='left') 

    df = df.merge(nat_hol[['date', 'is_national_holiday', 'holiday_type_nat']], on='date', how='left')
    df = df.merge(reg_hol[['date', 'state', 'is_regional_holiday', 'holiday_type_reg']], on=['date', 'state'], how='left')
    df = df.merge(loc_hol[['date', 'city', 'is_local_holiday', 'holiday_type_loc']], on=['date', 'city'], how='left')
    
    for col in ['is_national_holiday', 'is_regional_holiday', 'is_local_holiday']:
        df[col] = df[col].fillna(0).astype(int)

    df = df.merge(oil_only_dates, on='date', how='left')

    # 时间基础特征
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['dayofyear'] = df['date'].dt.dayofyear
    df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
    df['quarter'] = df['date'].dt.quarter
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)

    # 复活：细粒度时间段与地震影响特征
    df['month_period'] = pd.cut(df['day'], bins=[0, 10, 20, 32], labels=[0, 1, 2]).astype(int)
    df['is_rainy_season'] = ((df['month'] >= 12) | (df['month'] <= 5)).astype(int)
    df['days_since_earthquake'] = (df['date'] - earthquake_date).dt.days
    df['earthquake_effect'] = ((df['days_since_earthquake'] >= 0) & (df['days_since_earthquake'] <= 90)).astype(int)

    # 未来协变量（Promotion）：测试集已知，释放全量红利
    df['onpromotion'] = df['onpromotion'].fillna(0)
    for lag in [0, 1, 2, 3, 7]:
        df[f'onpromotion_lag_{lag}'] = df.groupby(['store_nbr', 'family'])['onpromotion'].shift(lag)

    # 过去协变量（Sales）：严格卡死16天安全平移锁
    grp = ['store_nbr', 'family']
    for lag in [16, 17, 18, 19, 20, 21, 28, 30, 60, 90, 180, 365]:
        df[f'sales_lag_{lag}'] = df.groupby(grp)['sales'].shift(lag)

    for w in [7, 14, 28, 60, 90, 180]:
        df[f'sales_rmean_{w}'] = df.groupby(grp)['sales'].transform(lambda x: x.shift(16).rolling(w, min_periods=1).mean())
    for w in [7, 14, 28]:
        df[f'sales_rstd_{w}'] = df.groupby(grp)['sales'].transform(lambda x: x.shift(16).rolling(w, min_periods=1).std())

    # 复活：强力的交叉特征编码
    cat_cols = ['family', 'city', 'state', 'type', 'holiday_type_nat', 'holiday_type_reg', 'holiday_type_loc']
    for col in cat_cols:
        df[col + '_le'] = LabelEncoder().fit_transform(df[col].astype(str))

    df['store_family'] = df['store_nbr'].astype(str) + '_' + df['family'].astype(str)
    df['store_family_le'] = LabelEncoder().fit_transform(df['store_family'])
    df['store_type'] = df['store_nbr'].astype(str) + '_' + df['type'].astype(str)
    df['store_type_le'] = LabelEncoder().fit_transform(df['store_type'])

    return df

# ---- 5. 标签状态注入与合并 ----
train['_is_train'] = 1
test['_is_train'] = 0
test['sales'] = np.nan

full = pd.concat([train, test], axis=0, ignore_index=True)
full = full.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)
full = build_all_features(full)

train_df = full[full['_is_train'] == 1].copy()
test_df = full[full['_is_train'] == 0].copy()

# ---- 6. 21天滑窗死商户判定 ----
max_train_date = train_df['date'].max()
recent_window = train_df[train_df['date'] > (max_train_date - pd.Timedelta(days=21))]
dead_bundles = recent_window.groupby(['store_nbr', 'family'])['sales'].sum().reset_index()
dead_bundles['is_dead'] = (dead_bundles['sales'] == 0).astype(int)

# 过滤纯字符列，防止 LightGBM 报错
drop_cols = ['id', 'date', 'sales', 'family', '_is_train', 'city', 'state', 'type', 
             'holiday_type_nat', 'holiday_type_reg', 'holiday_type_loc', 'store_family', 'store_type']
features = [c for c in train_df.columns if c not in drop_cols]

# ---- 7. 稳健全局重训 ----
val_cutoff = train_df['date'].max() - pd.Timedelta(days=15)
X_trn = train_df[train_df['date'] <= val_cutoff][features]
y_trn = np.log1p(train_df[train_df['date'] <= val_cutoff]['sales'].clip(lower=0))
X_val = train_df[train_df['date'] > val_cutoff][features]
y_val = np.log1p(train_df[train_df['date'] > val_cutoff]['sales'].clip(lower=0))

params = {
    'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
    'learning_rate': 0.05, 'num_leaves': 127, 'max_depth': 10,
    'min_child_samples': 50, 'feature_fraction': 0.7, 'bagging_fraction': 0.8,
    'bagging_freq': 5, 'n_jobs': -1, 'verbose': -1, 'seed': 42
}

dtrain = lgb.Dataset(X_trn, label=y_trn)
dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
model_val = lgb.train(params, dtrain, num_boost_round=2500, valid_sets=[dval], callbacks=[lgb.early_stopping(50)])

# 全量数据重训
print("⚙️ 正在执行全量健壮性重训...")
dtrain_full = lgb.Dataset(train_df[features], label=np.log1p(train_df['sales'].clip(lower=0)))
model_full = lgb.train(params, dtrain_full, num_boost_round=int(model_val.best_iteration * 1.15))

# 预测测试集
test_df['pred_sales'] = np.expm1(model_full.predict(test_df[features]))

# ---- 8. 后处理动态拦截 & 排序安全对齐 ----
test_df = test_df.merge(dead_bundles[['store_nbr', 'family', 'is_dead']], on=['store_nbr', 'family'], how='left')
test_df['is_dead'] = test_df['is_dead'].fillna(0)
test_df.loc[test_df['is_dead'] == 1, 'pred_sales'] = 0.0

# 终极拦截：测试集中的 12月25日 强制归零（虽然测试集只有16天不包含12月，但加上规则更严谨）
test_df.loc[(test_df['date'].dt.month == 12) & (test_df['date'].dt.day == 25), 'pred_sales'] = 0.0

submission = pd.DataFrame({
    'id': test_df['id'].astype(int),
    'sales': np.clip(test_df['pred_sales'].values, 0, None)
})

# 💥 关键修复：将提交格式按照原始 id 升序重新排列，确保与官方评测集行序绝对无缝对齐
submission = submission.sort_values('id').reset_index(drop=True)
submission.to_csv('submission.csv', index=False)

In [ ]:
# %% [markdown]
# # 🏪 Store Sales - Time Series Forecasting (Final High-Score Version)
# ## 期末专题 - 销量预测挑战赛
#
# **团队成员：**
# | 姓名 | 学号 |
# |------|------|
# | 郑泽凯 | 2025416010129 |
# | 邱婕芹 | 2025416010119 |
# | 何想 | 2025416010106 |

# %%
import numpy as np
import pandas as pd
import lightgbm as lgb
import os
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# ---- 1. 动态寻路 ----
DATA_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train.csv' in files:
        DATA_DIR = Path(root)
        break
if DATA_DIR is None:
    raise FileNotFoundError("❌ 未能在 /kaggle/input 下找到数据文件。")

train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['date'])
test = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=['date'])
stores = pd.read_csv(DATA_DIR / 'stores.csv')
oil = pd.read_csv(DATA_DIR / 'oil.csv', parse_dates=['date'])
holidays = pd.read_csv(DATA_DIR / 'holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv(DATA_DIR / 'transactions.csv', parse_dates=['date'])

# ---- 2. 完美时间轴重塑（消灭时间断层，防止 Shift 错位） ----
train_start = '2015-01-01'
train_end = train['date'].max()
test_start = test['date'].min()
test_end = test['date'].max()

all_dates = pd.date_range(train_start, test_end)
stores_list = train['store_nbr'].unique()
families_list = train['family'].unique()

multi_idx = pd.MultiIndex.from_product(
    [all_dates, stores_list, families_list],
    names=['date', 'store_nbr', 'family']
)
full = pd.DataFrame(index=multi_idx).reset_index()

# 合并训练与测试集基础销售数据
train_clean = train[train['date'] >= train_start]
full = full.merge(train_clean[['date', 'store_nbr', 'family', 'sales', 'onpromotion']], on=['date', 'store_nbr', 'family'], how='left')
full = full.merge(test[['date', 'store_nbr', 'family', 'onpromotion', 'id']], on=['date', 'store_nbr', 'family'], how='left', suffixes=('', '_test'))

full['onpromotion'] = full['onpromotion'].fillna(full['onpromotion_test']).fillna(0)
full['sales'] = full['sales'].fillna(0)  # 测试集及缺失日期闭店初始化为 0
full.drop(columns=['onpromotion_test'], inplace=True)

# ---- 3. 协变量前置加速处理（合并前计算，性能暴增 1000 倍） ----
# 日历时间
full['year'] = full['date'].dt.year
full['month'] = full['date'].dt.month
full['day'] = full['date'].dt.day
full['dayofweek'] = full['date'].dt.dayofweek
full['dayofyear'] = full['date'].dt.dayofyear
full['is_weekend'] = (full['dayofweek'] >= 5).astype(int)

# 发薪日连续流
paydays = all_dates[(all_dates.day == 15) | (all_dates.is_month_end)]
oil_only_dates = pd.DataFrame({'date': all_dates})
oil_only_dates['days_since_payday'] = oil_only_dates['date'].apply(lambda x: min([(x - p).days for p in paydays if (x - p).days >= 0], default=0))

# 外部油价
oil = oil.rename(columns={'dcoilwtico': 'oil_price'})
oil_full = pd.DataFrame({'date': all_dates}).merge(oil, on='date', how='left')
oil_full['oil_price'] = oil_full['oil_price'].interpolate(method='linear').bfill().ffill()
oil_full['oil_ma_7'] = oil_full['oil_price'].rolling(7, min_periods=1).mean()
oil_full['oil_ma_28'] = oil_full['oil_price'].rolling(28, min_periods=1).mean()
oil_only_dates = oil_only_dates.merge(oil_full, on='date', how='left')

# 国家节假日
nat_hol = holidays[(holidays['locale'] == 'National') & (holidays['transferred'] == False)].drop_duplicates('date')
nat_hol['is_national_holiday'] = 1
oil_only_dates = oil_only_dates.merge(nat_hol[['date', 'is_national_holiday']], on='date', how='left')
oil_only_dates['is_national_holiday'] = oil_only_dates['is_national_holiday'].fillna(0).astype(int)

full = full.merge(oil_only_dates, on='date', how='left')

# 门店与品类标签编码
stores_encoded = stores.copy()
for col in ['city', 'state', 'type', 'cluster']:
    stores_encoded[col] = LabelEncoder().fit_transform(stores_encoded[col].astype(str))
full = full.merge(stores_encoded, on='store_nbr', how='left')
full['family_le'] = LabelEncoder().fit_transform(full['family'].astype(str))

# 交易量历史对齐（安全滞后 16 天）
shifted_tx = transactions.copy()
shifted_tx['date'] = shifted_tx['date'] + pd.Timedelta(days=16)
shifted_tx = shifted_tx.rename(columns={'transactions': 'transactions_lag_16'})
full = full.merge(shifted_tx[['date', 'store_nbr', 'transactions_lag_16']], on=['date', 'store_nbr'], how='left')
full['transactions_lag_16'] = full['transactions_lag_16'].fillna(0)

# 促销已知特征全量 Lag
grp = ['store_nbr', 'family']
for lag in [1, 2, 3, 7, 14]:
    full[f'onpromo_lag_{lag}'] = full.groupby(grp)['onpromotion'].shift(lag).fillna(0)

# ---- 4. 销量历史动态特征初始化定义 ----
def update_sales_features(df):
    for lag in [1, 2, 3, 4, 5, 6, 7, 14, 21, 28]:
        df[f'sales_lag_{lag}'] = df.groupby(grp)['sales'].shift(lag)
    df['sales_rmean_7'] = df.groupby(grp)['sales'].transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    df['sales_rmean_28'] = df.groupby(grp)['sales'].transform(lambda x: x.shift(1).rolling(28, min_periods=1).mean())
    return df

print("⏳ 正在进行全局动态时序特征初始化...")
full = update_sales_features(full)

# 定义特征集
feature_cols = [
    'store_nbr', 'city', 'state', 'type', 'cluster', 'family_le',
    'year', 'month', 'day', 'dayofweek', 'dayofyear', 'is_weekend', 'days_since_payday',
    'oil_price', 'oil_ma_7', 'oil_ma_28', 'is_national_holiday',
    'onpromotion', 'onpromo_lag_1', 'onpromo_lag_2', 'onpromo_lag_3', 'onpromo_lag_7', 'onpromo_lag_14',
    'transactions_lag_16',
    'sales_lag_1', 'sales_lag_2', 'sales_lag_3', 'sales_lag_4', 'sales_lag_5', 'sales_lag_6', 'sales_lag_7', 'sales_lag_14', 'sales_lag_21', 'sales_lag_28',
    'sales_rmean_7', 'sales_rmean_28'
]

# ---- 5. 33品类并行特异性重训 ----
train_df = full[(full['date'] <= '2017-08-15') & (full['date'] >= '2015-02-01')].copy()
models_dict = {}
families = full['family'].unique()

print("🤖 正在为 33 个品类独立精细化拟合专属大模型...")
for fam in families:
    fam_train = train_df[train_df['family'] == fam]
    X = fam_train[feature_cols]
    y = np.log1p(fam_train['sales'])  # 对数处理对齐 RMSLE 评测标准
    
    model = lgb.LGBMRegressor(
        n_estimators=400,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    model.fit(X, y)
    models_dict[fam] = model

# ---- 6. 核心黑科技：自回归测试循环（Day-by-Day 动态特征赋能） ----
print("🔄 开启 16 天自回归递推滚动循环预测流...")
test_dates = sorted(full[full['date'] >= '2017-08-16']['date'].unique())

for current_date in test_dates:
    day_mask = full['date'] == current_date
    
    # 核心递归步：将前一天的预测销量作为最新的历史 Lag 特征刷新进来
    for lag in [1, 2, 3, 4, 5, 6, 7, 14, 21, 28]:
        prev_date = current_date - pd.Timedelta(days=lag)
        prev_sales = full[full['date'] == prev_date].set_index(['store_nbr', 'family'])['sales']
        full.loc[day_mask, f'sales_lag_{lag}'] = full[day_mask].set_index(['store_nbr', 'family']).index.map(prev_sales).values
        
    # 动态刷新滚动移动平均值
    lag_cols_7 = [full.loc[day_mask, f'sales_lag_{l}'] for l in range(1, 8)]
    full.loc[day_mask, 'sales_rmean_7'] = np.mean(lag_cols_7, axis=0)
    
    past_28_mask = (full['date'] < current_date) & (full['date'] >= current_date - pd.Timedelta(days=28))
    mean_28 = full[past_28_mask].groupby(['store_nbr', 'family'])['sales'].mean()
    full.loc[day_mask, 'sales_rmean_28'] = full[day_mask].set_index(['store_nbr', 'family']).index.map(mean_28).values

    # 调用各品类专属模型进行当日精准锁位预测
    for fam in families:
        fam_day_mask = (full['date'] == current_date) & (full['family'] == fam)
        if fam_day_mask.sum() == 0:
            continue
        X_test_day = full[fam_day_mask][feature_cols]
        
        pred_log = models_dict[fam].predict(X_test_day)
        pred_sales = np.expm1(pred_log)
        pred_sales = np.clip(pred_sales, 0, None)
        
        # 极为关键：将今天的预测值写回表格，供明天做 Lag_1 特征时读取！
        full.loc[fam_day_mask, 'sales'] = pred_sales

# ---- 7. 21天最新死商户束滑窗拦截 ----
print("🎄 执行最新 21 天彻底下架/断货动态零销售后处理规则...")
recent_active_window = full[(full['date'] >= '2017-07-26') & (full['date'] <= '2017-08-15')]
bundle_sales = recent_active_window.groupby(['store_nbr', 'family'])['sales'].sum()
dead_bundles = bundle_sales[bundle_sales == 0].index

test_results = full[full['date'] >= '2017-08-16'].copy()
test_results = test_results.set_index(['store_nbr', 'family'])
test_results.loc[test_results.index.isin(dead_bundles), 'sales'] = 0.0
test_results = test_results.reset_index()

# 严格按照原始测试集 id 顺序升序对齐输出，彻底隔绝行错位风险
submission = test[['id']].merge(test_results[['id', 'sales']], on='id', how='left')
submission['sales'] = submission['sales'].fillna(0)
submission.to_csv('submission.csv', index=False)
print("🏁 恭喜！")